# Loading a raw UFO model

This tutorial crosses FeynKit's optional UFO boundary: a conventional Python UFO package is normalized into the same typed `Model` used by the core diagram and expression APIs. We use a tiny scalar theory so every diagnostic count is easy to inspect.

## Requirements

Raw UFO import requires **Python 3.11 or newer** and `ufo-model-loader>=0.1.6`. Install the supported extra with:

```bash
pip install 'symbolica[feynkit-ufo]'
```

This is an explicit optional workflow. The normalized JSON tutorials `00`–`03` need only the base `symbolica` package.

In [ ]:
import os
from pathlib import Path

import symbolica.community.feynkit as fk

DATA = next(path for path in (Path("data"), Path("examples/feynkit/data")) if path.exists())
UFO_MODEL = (DATA / "ufo_scalars").resolve()

## Normalize the UFO package

`UfoLoader.load` returns three typed results together: `model` for physics operations, `parameters` for the applied numerical inputs, and `diagnostics` recording exactly how normalization was requested. `restriction_name="default"` applies `restrict_default.dat`.

The environment variable below belongs to this teaching fixture—not to FeynKit generally. It limits this generated scalar model to two- and three-point interactions, keeping the tutorial fast and deterministic.

In [ ]:
interaction_key = "UFO_SCALARS_MODEL_N_POINT_INTERACTIONS"
previous_interactions = os.environ.get(interaction_key)
os.environ[interaction_key] = "2,3"
try:
    loaded = fk.UfoLoader(restriction_name="default").load(UFO_MODEL)
finally:
    if previous_interactions is None:
        os.environ.pop(interaction_key, None)
    else:
        os.environ[interaction_key] = previous_interactions

model = loaded.model
parameters = loaded.parameters
diagnostics = loaded.diagnostics

## Check normalization diagnostics

These counts describe the normalized model actually handed to Rust. They are useful provenance when comparing model restrictions or diagnosing an unexpected set of interaction rules.

In [ ]:
counts = {
    "orders": diagnostics.order_count,
    "model_parameters": diagnostics.model_parameter_count,
    "particles": diagnostics.particle_count,
    "propagators": diagnostics.propagator_count,
    "lorentz_structures": diagnostics.lorentz_structure_count,
    "couplings": diagnostics.coupling_count,
    "vertices": diagnostics.vertex_rule_count,
    "functions": diagnostics.function_count,
    "form_factors": diagnostics.form_factor_count,
    "parameter_values": diagnostics.parameter_value_count,
}
{
    "counts": counts,
    "source": diagnostics.source,
    "restriction": diagnostics.restriction_name,
    "simplified": diagnostics.simplify_model,
    "wrapped_lorentz_indices": diagnostics.wrap_indices_in_lorentz_structures,
}

## Inspect the typed model and restriction card

The loader has now left the Python UFO object model behind. Particle, parameter, coupling, and vertex lookups use FeynKit's typed API. Zero widths in the restriction card become internal zero parameters during simplification, leaving the coupling and two nonzero masses as independent numerical inputs.

In [ ]:
particle_rows = [
    {
        "name": particle.name,
        "pdg": particle.pdg_code,
        "mass_parameter": particle.mass_parameter,
        "massless": particle.is_massless,
    }
    for particle in model.particles
]
particle_rows

In [ ]:
parameter_values = dict(parameters.items())
parameter_values

## Reuse the model in diagram generation

Nothing downstream is UFO-specific. Generate diagrams directly from the loaded `model`; selectors, generation options, graph inspection, CFF construction, and Symbolica expressions all compose unchanged.

In [ ]:
options = fk.GenerationOptions(max_vertices=3)
options.add_vertex_allow(["V_3_SCALAR_000"])

generated = model.generate_diagrams(
    incoming=["scalar_0"],
    outgoing=["scalar_0", "scalar_0"],
    loops=0,
    options=options,
)
{
    "diagrams": len(generated.diagrams),
    "topologies_considered": generated.report.topology_count,
    "first_diagram": generated.diagrams[0].name,
}

## Working with your own UFO

Point `UfoLoader.load` at the UFO package directory and choose a `restriction_name` matching `restrict_<name>.dat`; use `None` to accept the package's default behavior. Keep `simplify_model=True` for the usual removal of zero contributions. Disable `wrap_indices_in_lorentz_structures` only when interoperating with code that specifically expects unwrapped UFO syntax.

For reproducible production workflows, normalize once with the UFO bridge and serialize the resulting `Model` to JSON; later sessions can load that JSON without the optional Python dependency.